# RE-THREAD: Reusable ResNet-18 Vision Classifier Training

**Team Circular Transformation · PCCOE Pune**  
**Purpose:** Train a lightweight, ImageNet-pretrained ResNet-18 classifier for:
- **Task A: Garment Condition** (`new`, `gently_used`, `worn`, `heavily_worn`)
- **Task B: Fabric Material** (`natural`, `synthetic`, `blend`)

---

### Key Machine Learning Principles in this Notebook:
1. **Pretrained Backbone:** Starts with ImageNet-pretrained weights (`ResNet18_Weights.DEFAULT`) for fast convergence on small datasets.
2. **Group Split Strategy:** Prevents data leakage by ensuring all multi-angle photos of the same physical garment remain strictly in the same split (train OR val, never both).
3. **Class Balancing:** Computes inverse-frequency class weights so unbalanced defect/wear categories receive fair gradient updates.
4. **Holistic Evaluation:** Measures both overall Accuracy and Macro-F1, alongside a Confusion Matrix to inspect adjacent wear misclassifications.
5. **Artifact Output:** Exports model weights directly to Google Drive as `condition.pt` or `material.pt`.

## Step 1: Environment Setup & GPU Check

In [ ]:
!nvidia-smi

import torch
import torchvision
print(f"PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Device: {device}")

## Step 2: Mount Google Drive
Saves trained weights persistently to your Google Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/rethread_weights'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Trained weights will be exported to: {OUTPUT_DIR}")

## Step 3: Configure Training Task
Select whether you are training **Task A (Condition)** or **Task B (Material)**.

In [ ]:
# Choose TASK: 'condition' or 'material'
TASK = "condition"  # Change to 'material' when training material classifier

if TASK == "condition":
    CLASS_NAMES = ["new", "gently_used", "worn", "heavily_worn"]
    MODEL_SAVE_NAME = "condition.pt"
elif TASK == "material":
    CLASS_NAMES = ["natural", "synthetic", "blend"]
    MODEL_SAVE_NAME = "material.pt"
else:
    raise ValueError(f"Unknown task: {TASK}")

print(f"Configured Task: {TASK.upper()}")
print(f"Target Classes: {CLASS_NAMES}")
print(f"Output Weights File: {MODEL_SAVE_NAME}")

## Step 4: Dataset Folder Structure & Group Split Strategy

### Documented Folder Structure:
```text
dataset/
  train/
    new/            (or natural/)
      garment01_front.jpg
      garment01_texture.jpg
    gently_used/    (or synthetic/)
      garment02_front.jpg
    worn/           (or blend/)
    heavily_worn/
  val/
    new/
    gently_used/
    worn/
    heavily_worn/
```

### Group Split Strategy (Crucial to Prevent Data Leakage):
If a user photographs one shirt from 3 angles (`front`, `texture`, `damage`), those 3 images share the same fabric, background, and lighting. If one goes into train and two into validation, the model will simply memorize the garment!  
The code below demonstrates grouping by `garment_id` using `GroupShuffleSplit`:

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# Demonstration of group splitting logic:
# If your image filenames follow 'garmentID_angle.jpg':
sample_files = [
    "g001_front.jpg", "g001_texture.jpg", "g001_detail.jpg",
    "g002_front.jpg", "g002_texture.jpg",
    "g003_front.jpg", "g003_texture.jpg", "g003_label.jpg",
    "g004_front.jpg",
]
garment_ids = [f.split("_")[0] for f in sample_files]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(sample_files, groups=garment_ids))

print(f"Sample Group Split: {len(train_idx)} train images, {len(val_idx)} val images.")
print(f"Train garments: {set([garment_ids[i] for i in train_idx])}")
print(f"Val garments: {set([garment_ids[i] for i in val_idx])}")
print("Notice: No garment appears in both sets! Zero data leakage.")

## Step 5: Data Augmentation & Loaders

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Standard ImageNet normalization statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Set DATASET_DIR to your dataset location
DATASET_DIR = f"/content/dataset_{TASK}"

# Example loader creation (uncomment when dataset is mounted):
# train_dataset = ImageFolder(os.path.join(DATASET_DIR, "train"), transform=train_transforms)
# val_dataset = ImageFolder(os.path.join(DATASET_DIR, "val"), transform=val_transforms)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

## Step 6: Model Construction (ResNet-18 with Replaced FC Layer)

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

def build_resnet18_classifier(num_classes: int):
    # 1. Load pretrained ImageNet ResNet-18
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    
    # 2. Replace final fully connected layer (model.fc)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model

model = build_resnet18_classifier(len(CLASS_NAMES)).to(device)
print(f"Built ResNet-18 with {len(CLASS_NAMES)} output classes: {CLASS_NAMES}")

## Step 7: Class Balancing Loss & Training Loop

In [ ]:
import torch.optim as optim
import numpy as np

# Compute balanced class weights:
# weight_i = total_samples / (num_classes * count_i)
# class_weights = torch.tensor([...], dtype=torch.float).to(device)
# criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

print("Optimizer and loss initialized.")

## Step 8: Training & Validation Execution with Macro-F1 & Confusion Matrix

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(dataloader.dataset)

def validate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    return acc, macro_f1, cm

## Step 9: Save Weights to Google Drive (torch.save)
Saves the trained model state and class names mapping.

In [ ]:
save_payload = {
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "num_classes": len(CLASS_NAMES),
    "task": TASK,
}

target_save_path = os.path.join(OUTPUT_DIR, MODEL_SAVE_NAME)
torch.save(save_payload, target_save_path)
print(f"SUCCESS: Saved trained {TASK} model to Google Drive at:\n{target_save_path}")
print("\n--- HOW TO DEPLOY TO RE-THREAD BACKEND ---")
print(f"1. Download '{MODEL_SAVE_NAME}' from Google Drive.")
print(f"2. Place in: re-thread/weights/{MODEL_SAVE_NAME}")
print("3. Restart the FastAPI backend. It will load weights automatically at startup!")